# 01 · Collect charts, store pages, and publisher sites

Runs the three collection stages on the snapshot date. Everything here contacts external sites, so run it **once**, on the snapshot date, after:

1. `.env` has `SNAPSHOT_DATE`, `SCRAPER_CONTACT`, and `REQUEST_DELAY_SECONDS`;
2. each store's terms are checked and `config/sources.json` is updated (`terms_allow_automation`, `verified`, console chart URLs and selectors);
3. any chart-tracker exports are saved in `data/interim/tracker_charts/`.

Raw responses go to `data/raw/<snapshot>/` (private, gitignored). Every record carries its snapshot date, source, retrieval time, and evidence path.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

## Stage 1 · Best-seller charts
Store collector first; tracker export if that fails; a logged gap if neither works. `charts_status.csv` records which method each platform-market used.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

from scrapers.charts.run import main as collect_charts
charts = collect_charts()

import pandas as pd
from scrapers.records import INTERIM
pd.read_csv(INTERIM / "charts_status.csv")

## Stage 2 · Store pages
App Store via the iTunes Lookup API; Steam via the appdetails API (Belgium in Dutch, French, and German); consoles via Playwright on public product pages.

In [ ]:
from scrapers.store_pages.run import main as collect_store_pages
collect_store_pages()

from scrapers.records import read_jsonl
store = pd.DataFrame(read_jsonl(INTERIM / "store_pages.jsonl"))
store.groupby(["platform", "market"]).size().unstack(fill_value=0)

## Stage 3a · Publisher seeds
Builds `config/publisher_seeds.csv` from store links. **Stop here and review it by hand** (remove storefront and social links, add support sites, set `reviewed` to `yes`) before crawling.

In [ ]:
from scrapers.publisher.seeds import build
build()

## Stage 3b · Publisher crawl
Breadth-first, once per market, robots.txt obeyed, no cookies. Scrapy's reactor can only start once per kernel, so this runs as a separate process.

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "scrapers.publisher.crawl"], cwd=ROOT, check=True)
pages = pd.DataFrame(read_jsonl(INTERIM / "publisher_pages.jsonl"))
pages.groupby(["seed_id", "market"]).agg(pages=("url", "size"), max_depth=("depth", "max"),
                                          odds_pages=("odds_terms", lambda s: sum(bool(x) for x in s)))

## Archive odds pages
Saves every page with odds terms to the Wayback Machine. Safe to re-run: archived URLs are skipped.

In [ ]:
from scrapers.archive import main as archive
archive()
pd.read_csv(INTERIM / "archives.csv")["status"].value_counts()

## Wrong turns

_Keep failed approaches in this notebook. For each one, leave the code in place and add a markdown cell saying what was tried, what went wrong, and how it changed the next step._